<h1 style='color:#0f766e' align='center'>SmartNest Rental Intelligence - Model Training</h1>

Dataset: House_Rent_Dataset_TN_synthetic_text_1500_preprocessed.csv

In [4]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [5]:
DATA_PATH = Path.cwd() / 'House_Rent_Dataset_TN_synthetic_text_1500_preprocessed.csv'
ARTIFACT_DIR = Path.cwd().parent / 'server' / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COLS = [
    'Listing_Title',
    'Listing_Description',
    'Amenities_Text',
    'Landmarks_Text',
    'Reviews_Text',
    'Complaints_Text',
    'Lease_Rules_Text',
    'Inquiry_Text',
    'Inspection_Notes_Text',
]

<h2 style='color:#0f766e'>Load and clean data</h2>

In [6]:
df = pd.read_csv(DATA_PATH)
for col in ['Size', 'BHK', 'Bathroom', 'Rent']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=['Size', 'BHK', 'Bathroom', 'Rent'])
df['Scam_Flag'] = df['Scam_Flag'].astype(str).str.lower().map({'yes': 1, 'no': 0})
df['Scam_Flag'] = df['Scam_Flag'].fillna(0).astype(int)
df.shape

(5000, 23)

<h2 style='color:#0f766e'>Save dropdown options</h2>

In [7]:
options = {
    'cities': sorted(df['City'].dropna().unique().tolist()),
    'area_types': sorted(df['Area Type'].dropna().unique().tolist()),
    'furnishing_statuses': sorted(df['Furnishing Status'].dropna().unique().tolist()),
    'tenant_preferred': sorted(df['Tenant Preferred'].dropna().unique().tolist()),
}
(ARTIFACT_DIR / 'options.json').write_text(json.dumps(options, indent=2))

599

<h2 style='color:#0f766e'>Rent prediction - Random Forest Regressor</h2>

In [8]:
rent_features = ['Size', 'BHK', 'Bathroom', 'City', 'Area Type', 'Furnishing Status', 'Tenant Preferred']
X_rent = df[rent_features].copy()
y_rent = df['Rent'].copy()
X_rent = pd.get_dummies(X_rent, columns=['City', 'Area Type', 'Furnishing Status', 'Tenant Preferred'])

X_train, X_test, y_train, y_test = train_test_split(X_rent, y_rent, test_size=0.2, random_state=42)
rent_model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rent_model.fit(X_train, y_train)

preds = rent_model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)
print(f'Rent RF MAE={mae:.2f} RMSE={rmse:.2f} R2={r2:.3f}')

joblib.dump(rent_model, ARTIFACT_DIR / 'rent_model.pkl')
(ARTIFACT_DIR / 'rent_columns.json').write_text(json.dumps({'columns': X_rent.columns.tolist()}, indent=2))

Rent RF MAE=6316.54 RMSE=21816.06 R2=0.244


793

<h2 style='color:#0f766e'>Fake listing detection - Random Forest + Logistic Regression</h2>

In [9]:
scam_features = ['Rent', 'Size', 'BHK', 'Bathroom', 'City', 'Area Type', 'Furnishing Status', 'Tenant Preferred']
X_scam = df[scam_features].copy()
y_scam = df['Scam_Flag'].copy()
X_scam = pd.get_dummies(X_scam, columns=['City', 'Area Type', 'Furnishing Status', 'Tenant Preferred'])

X_train, X_test, y_train, y_test = train_test_split(X_scam, y_scam, test_size=0.2, random_state=42, stratify=y_scam)

lr_model = LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear')
rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')

lr_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

print(f'Scam LR Accuracy={lr_model.score(X_test, y_test):.3f} | Scam RF Accuracy={rf_model.score(X_test, y_test):.3f}')

joblib.dump(lr_model, ARTIFACT_DIR / 'scam_lr_model.pkl')
joblib.dump(rf_model, ARTIFACT_DIR / 'scam_rf_model.pkl')
(ARTIFACT_DIR / 'scam_columns.json').write_text(json.dumps({'columns': X_scam.columns.tolist()}, indent=2))

Scam LR Accuracy=0.588 | Scam RF Accuracy=0.883


805

<h2 style='color:#0f766e'>Text verification - Naive Bayes</h2>

In [10]:
df['text_all'] = df[TEXT_COLS].fillna('').agg(' '.join, axis=1)
X_train, X_test, y_train, y_test = train_test_split(df['text_all'], df['Scam_Flag'], test_size=0.2, random_state=42, stratify=df['Scam_Flag'])

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

text_model = MultinomialNB()
text_model.fit(X_train_vec, y_train)
print(f'Text NB Accuracy={text_model.score(X_test_vec, y_test):.3f}')

joblib.dump(text_model, ARTIFACT_DIR / 'text_nb_model.pkl')
joblib.dump(vectorizer, ARTIFACT_DIR / 'text_vectorizer.pkl')

Text NB Accuracy=0.887


['d:\\smartnest_rental_platform\\server\\artifacts\\text_vectorizer.pkl']

<h2 style='color:#0f766e'>Recommendation - KNN Similarity</h2>

In [11]:
reco_features = ['Size', 'BHK', 'Bathroom', 'Rent', 'City', 'Area Type', 'Furnishing Status', 'Tenant Preferred']
X_reco = df[reco_features].copy()

reco_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Size', 'BHK', 'Bathroom', 'Rent']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['City', 'Area Type', 'Furnishing Status', 'Tenant Preferred']),
    ]
)

X_reco_proc = reco_preprocessor.fit_transform(X_reco)
reco_nn = NearestNeighbors(metric='cosine', n_neighbors=6)
reco_nn.fit(X_reco_proc)

summary_cols = ['City', 'Area Locality', 'Rent', 'Size', 'BHK', 'Bathroom', 'Furnishing Status', 'Tenant Preferred']
df[summary_cols].to_csv(ARTIFACT_DIR / 'listing_summary.csv', index=False)

joblib.dump(reco_preprocessor, ARTIFACT_DIR / 'reco_preprocessor.pkl')
joblib.dump(reco_nn, ARTIFACT_DIR / 'reco_nn_model.pkl')
joblib.dump(X_reco_proc, ARTIFACT_DIR / 'reco_matrix.pkl')

['d:\\smartnest_rental_platform\\server\\artifacts\\reco_matrix.pkl']

<h2 style='color:#0f766e'>Locality analysis - K-Means</h2>

In [12]:
city_stats = (
    df.groupby('City')
    .agg(
        avg_rent=('Rent', 'mean'),
        avg_size=('Size', 'mean'),
        avg_bhk=('BHK', 'mean'),
        avg_bath=('Bathroom', 'mean'),
        listings=('Rent', 'count'),
    )
    .reset_index()
)

features = city_stats[['avg_rent', 'avg_size', 'avg_bhk', 'avg_bath']]
scaler = StandardScaler()
scaled = scaler.fit_transform(features)

n_clusters = max(1, min(4, len(city_stats)))
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
city_stats['cluster'] = kmeans.fit_predict(scaled)

joblib.dump(kmeans, ARTIFACT_DIR / 'locality_kmeans.pkl')
joblib.dump(scaler, ARTIFACT_DIR / 'locality_scaler.pkl')
city_stats.to_csv(ARTIFACT_DIR / 'locality_city_stats.csv', index=False)
city_stats.head()

,City,avg_rent,avg_size,avg_bhk,avg_bath,listings,cluster
0,chennai,24357.928155,992.255340,2.068447,1.798544,2060,3
1,coimbatore,24563.580609,1018.359639,2.098083,1.847802,887,3
2,cuddalore,19698.412698,1010.666667,2.079365,1.730159,63,0
3,dindigul,18172.727273,937.779221,1.974026,1.662338,77,2
4,erode,22607.272727,986.000000,2.066667,1.703030,165,0


In [13]:
print('Artifacts saved in:', ARTIFACT_DIR)

Artifacts saved in: d:\smartnest_rental_platform\server\artifacts
